# Zestawienie wyników wszystkich modeli

Notatnik laduje CSV-ki z `wyniki/` (jeden CSV per model) i generuje:
- wspólną tabelę porównawczą
- wykresy słupkowe per metryka
- heatmapy macierzy pomyłek

Każdy CSV w `wyniki/` powinien mieć format:
```
model,wariant,tryb,acc,macro_f1,weighted_f1,precision_macro,recall_macro
```

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

WYNIKI_DIR = 'wyniki'
csv_files = [f for f in os.listdir(WYNIKI_DIR) if f.endswith('.csv')]
print('Znalezione CSV:', csv_files)

In [ ]:
dfs = [pd.read_csv(os.path.join(WYNIKI_DIR, f)) for f in csv_files]
if dfs:
    wyniki = pd.concat(dfs, ignore_index=True)
    print(wyniki)
else:
    print('Brak CSV-ek w wyniki/. Zapisz wyniki z notebookow jako CSV i wroc tutaj.')
    wyniki = pd.DataFrame()

## Tabela porównawcza

In [ ]:
if not wyniki.empty:
    ranking = (wyniki
               .sort_values(['wariant', 'macro_f1'], ascending=[True, False])
               .round(4)
               .reset_index(drop=True))
    display(ranking)

## Wykres słupkowy macro-F1 per model

In [ ]:
if not wyniki.empty:
    for wariant, sub in wyniki.groupby('wariant'):
        sub = sub.sort_values('macro_f1', ascending=True)
        fig, ax = plt.subplots(figsize=(9, 0.4 * len(sub) + 1.5))
        ax.barh(sub['model'] + ' (' + sub['tryb'] + ')', sub['macro_f1'])
        ax.set_xlabel('macro-F1')
        ax.set_title(f'Porownanie modeli - wariant {wariant}')
        ax.grid(axis='x', alpha=0.3)
        for i, v in enumerate(sub['macro_f1']):
            ax.text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=9)
        plt.tight_layout()
        plt.savefig(os.path.join(WYNIKI_DIR, f'porownanie_{wariant}.png'), dpi=120)
        plt.show()